# Problem 2 – Part 1: Data Cleaning (Python)
**Goal:** turn both raw files into clean tables with **one row per encounter**, so Task 1 and Task 2 can be answered in **SQL**.

**Encounter (visit) = Patient + Provider + Date of Service.**

| Output file | One row = | Columns |
|---|---|---|
| `ehr_encounters.csv` | one visit in the clinic's EHR | patient, provider, dos, cpt_codes, n_codes |
| `db_encounters.csv` | one visit in Normandy (our DB) | patient, provider, dos, cpt_codes, n_codes, db_records |
| `ehr_codes.csv` | one code on one EHR visit | patient, provider, dos, cpt |
| `db_codes.csv` | one code on one DB visit | patient, provider, dos, cpt |

The two `*_codes` files let SQL answer code questions ("which codes never reach the DB?") with a simple JOIN / IN.

## 1. Load the raw files

In [128]:
import pandas as pd

ehr = pd.read_csv("../Case Study/Ops Case Study Dataset - Sample EHR Data (3).csv")
db  = pd.read_csv("../Case Study/Ops Case Study Dataset - Sample DB Data (3).csv")

print("EHR rows:", len(ehr))
print("DB rows :", len(db))

EHR rows: 25956
DB rows : 6482


In [129]:
ehr.head()

,Patient Name,Provider Name,Date of Service,CPT Code
0,Ava Johnson,Liam Young,09/24/2024,97010
1,Ava Johnson,Liam Young,09/24/2024,97140
2,Sophia King,Sebastian Miller,07/03/2024,97140
3,Carter Brown,Charlotte Williams,09/09/2024,97014
4,Olivia Davis,Noah Lee,09/09/2024,97140


In [130]:
db.head()

,Patient Name,Provider Name,from_date_range,cpt_codes
0,Lucas Smith,Aiden King,2024-07-03,"{97110,97112,97140,97010}"
1,Lucas Smith,Charlotte Williams,2024-07-11,"{97112,97010,97140,97110}"
2,Lucas Smith,Charlotte Williams,2024-07-17,"{97110,97140,97112,97010}"
3,Lucas Smith,Julian Lee,2024-07-27,"{97140,97110,97112,97010}"
4,Lucas Smith,Aiden King,2024-08-07,"{97010,97110,97112,97140}"


**What we see – the two files have different shapes:**
- **EHR:** one row per **code**. A visit with 4 codes = 4 rows. Date like `09/24/2024`.
- **DB:** one row per **visit**, all codes packed in one text `{97110,97112}`. Date like `2024-09-24`.

**Plan:** fix names, blanks, dates and spaces → then bring **both** to one row per visit, with the codes written the same way (`"97010, 97110, 97140"`).

## 2. Same column names

In [131]:
ehr = ehr.rename(columns={"Patient Name": "patient", "Provider Name": "provider",
                          "Date of Service": "dos", "CPT Code": "cpt"})
db  = db.rename(columns={"Patient Name": "patient", "Provider Name": "provider",
                         "from_date_range": "dos", "cpt_codes": "cpt_list"})
print(ehr.columns.tolist())
print(db.columns.tolist())

['patient', 'provider', 'dos', 'cpt']
['patient', 'provider', 'dos', 'cpt_list']


## 3. Remove blank rows

In [132]:
print("Blanks in EHR:\n", ehr.isna().sum(), "\n")
print("Blanks in DB:\n", db.isna().sum())

Blanks in EHR:
 patient     0
provider    2
dos         2
cpt         2
dtype: int64 

Blanks in DB:
 patient     0
provider    0
dos         0
cpt_list    0
dtype: int64


In [133]:
# 2 EHR rows have no provider, date or code -> not a real visit, drop them
print(ehr[ehr["provider", "dos"].isna()])
ehr = ehr.dropna()
print("EHR rows after drop:", len(ehr))

KeyError: ('provider', 'dos')

## 4. Same date format

In [ ]:
ehr["dos"] = pd.to_datetime(ehr["dos"], format="%m/%d/%Y")
db["dos"]  = pd.to_datetime(db["dos"],  format="%Y-%m-%d")

print("EHR:", ehr["dos"].min().date(), "to", ehr["dos"].max().date())
print("DB :", db["dos"].min().date(),  "to", db["dos"].max().date())

EHR: 2024-07-01 to 2024-09-30
DB : 2024-07-01 to 2024-09-30


## 5. Trim spaces in text columns

In [ ]:
for col in ["patient", "provider", "cpt"]:
    ehr[col] = ehr[col].str.strip()
for col in ["patient", "provider", "cpt_list"]:
    db[col] = db[col].str.strip()

## 6. DB: unpack the code text so both files list codes the same way
`{97110,97112}` → remove `{ }` → split by comma → one code per row.
(We unpack first so we can sort the codes and merge duplicate DB records in the next step.)

In [ ]:
db["cpt"] = db["cpt_list"].str.strip("{}").str.split(",")
db_codes = db.explode("cpt")
db_codes["cpt"] = db_codes["cpt"].str.strip()

db_codes[["patient", "provider", "dos", "cpt"]].head(8)

,patient,provider,dos,cpt
0,Lucas Smith,Aiden King,2024-07-03,97110
0,Lucas Smith,Aiden King,2024-07-03,97112
0,Lucas Smith,Aiden King,2024-07-03,97140
0,Lucas Smith,Aiden King,2024-07-03,97010
1,Lucas Smith,Charlotte Williams,2024-07-11,97112
1,Lucas Smith,Charlotte Williams,2024-07-11,97010
1,Lucas Smith,Charlotte Williams,2024-07-11,97140
1,Lucas Smith,Charlotte Williams,2024-07-11,97110


## 7. Build ONE ROW PER ENCOUNTER
For each visit (patient + provider + date):
- keep each code once (`drop_duplicates`)
- sort the codes, so the text is always in the same order
- join them into one text: `"97010, 97110, 97140"`
- count the codes

In [ ]:
key = ["patient", "provider", "dos"]

# ---- EHR ----
ehr_codes = ehr[key + ["cpt"]].drop_duplicates().sort_values("cpt")
print("EHR duplicate code lines removed:", len(ehr) - len(ehr_codes))

ehr_enc = (ehr_codes.groupby(key)
           .agg(cpt_codes=("cpt", ", ".join), n_codes=("cpt", "count"))
           .reset_index())
print("EHR:", len(ehr), "code rows ->", len(ehr_enc), "encounters")
ehr_enc.head()

EHR duplicate code lines removed: 331
EHR: 25954 code rows -> 6612 encounters


,patient,provider,dos,cpt_codes,n_codes
0,Aiden Allen,Aiden King,2024-07-03,97113,1
1,Aiden Allen,Aiden King,2024-07-10,97113,1
2,Aiden Allen,Aiden King,2024-07-15,97113,1
3,Aiden Allen,Aiden King,2024-07-24,97113,1
4,Aiden Allen,Aiden King,2024-07-31,97113,1


In [ ]:
# ---- DB ----
# Some visits appear in more than one DB record -> count them before merging
records_per_visit = db.groupby(key).size().reset_index(name="db_records")
print("DB visits stored in more than one record:", (records_per_visit["db_records"] > 1).sum())

db_codes = db_codes[key + ["cpt"]].drop_duplicates().sort_values("cpt")
db_enc = (db_codes.groupby(key)
          .agg(cpt_codes=("cpt", ", ".join), n_codes=("cpt", "count"))
          .reset_index())
db_enc = db_enc.merge(records_per_visit, on=key)
print("DB:", len(db), "records ->", len(db_enc), "encounters")
db_enc.head()

DB visits stored in more than one record: 108
DB: 6482 records -> 6374 encounters


,patient,provider,dos,cpt_codes,n_codes,db_records
0,Aiden Allen,Aiden King,2024-07-03,97113,1,1
1,Aiden Allen,Aiden King,2024-07-10,97113,1,1
2,Aiden Allen,Aiden King,2024-07-15,97113,1,1
3,Aiden Allen,Aiden King,2024-07-24,97113,1,1
4,Aiden Allen,Aiden King,2024-07-31,97113,1,1


## 8. Quick checks, then export

In [ ]:
# Each visit must appear only once in each table
print("EHR duplicate visits:", ehr_enc.duplicated(key).sum())
print("DB  duplicate visits:", db_enc.duplicated(key).sum())
print("Patients  EHR / DB:", ehr_enc["patient"].nunique(), "/", db_enc["patient"].nunique())
print("Providers EHR / DB:", ehr_enc["provider"].nunique(), "/", db_enc["provider"].nunique())

EHR duplicate visits: 0
DB  duplicate visits: 0
Patients  EHR / DB: 542 / 537
Providers EHR / DB: 8 / 8


In [ ]:
# Dates as YYYY-MM-DD text -> PostgreSQL reads it as a DATE
ehr_enc["dos"] = ehr_enc["dos"].dt.strftime("%Y-%m-%d")
db_enc["dos"]  = db_enc["dos"].dt.strftime("%Y-%m-%d")

ehr_enc.to_csv("ehr_encounters.csv", index=False)
db_enc.to_csv("db_encounters.csv", index=False)
print("Saved ehr_encounters.csv (", len(ehr_enc), "rows ) and db_encounters.csv (", len(db_enc), "rows )")

Saved ehr_encounters.csv ( 6612 rows ) and db_encounters.csv ( 6374 rows )


In [ ]:
# Code-level files (one row per visit + code, each code once per visit)
ehr_codes["dos"] = ehr_codes["dos"].dt.strftime("%Y-%m-%d")
db_codes["dos"]  = db_codes["dos"].dt.strftime("%Y-%m-%d")

ehr_codes.to_csv("ehr_codes.csv", index=False)
db_codes.to_csv("db_codes.csv", index=False)
print("Saved ehr_codes.csv (", len(ehr_codes), "rows ) and db_codes.csv (", len(db_codes), "rows )")

Saved ehr_codes.csv ( 25623 rows ) and db_codes.csv ( 24470 rows )


**Next:** load both files into PostgreSQL / Retool Database and run `p2_task1_task2.sql` (or, in Retool, upload all 4 CSVs and run `p2_retool_setup.sql`).